In [1]:
import pandas as pd             # data package
import matplotlib.pyplot as plt # graphics 
import datetime as dt
import numpy as np

import requests, io             # internet and input tools  
import zipfile as zf            # zip file tools 
import os  

#import weightedcalcs as wc
#import numpy as np

import pyarrow as pa
import pyarrow.parquet as pq

In [2]:
date = "current"

my_key = "&key=34e40301bda77077e24c859c6c6c0b721ad73fc7"
# This is my key. I'm nice and I have it posted. If you will be doing more with this
# please get your own key!

In [3]:
end_use = "naics?get=CON_VAL_MO,CTY_CODE,CTY_NAME,SUMMARY_LVL"

url = "https://api.census.gov/data/timeseries/intltrade/imports/" + end_use 
url = url + my_key + "&time==from+2013-01"

r = requests.get(url) 
    
print(r)
    
df = pd.DataFrame(r.json()[1:]) # This then converts it to a dataframe
    # Note that the first entry is the labels

df.columns = r.json()[0]

df["total_imports"] = df["CON_VAL_MO"].astype(float)

df = df[df.SUMMARY_LVL == "DET"]

grp = df.groupby(["CTY_NAME"])

top_products = grp.agg({"total_imports":"sum","CTY_CODE":"first"})

country_list = list(top_products.sort_values(by = "total_imports", ascending = False).CTY_CODE)[0:31]


['TOTAL FOR ALL COUNTRIES','NAFTA','EUROPEAN UNION']

<Response [200]>


['TOTAL FOR ALL COUNTRIES', 'NAFTA', 'EUROPEAN UNION']

In [4]:
country_list[0] = ""

In [5]:
country_list.extend(["0003", "0020"])

In [6]:
len(country_list)

33

In [7]:
end_use = "hs?get=CTY_NAME,CON_VAL_MO,CAL_DUT_MO,I_COMMODITY,I_COMMODITY_SDESC"

surl = "https://api.census.gov/data/timeseries/intltrade/imports/" + end_use 

surl  = surl + my_key + "&time=" + "from+2013-01" + "&COMM_LVL=HS10" 

for xxx in country_list:
    
    out_file = ".\\data"+ "\\imports-hs10\\" + xxx + "data-" + date + ".parquet"
    
    if xxx == "":
        out_file = ".\\data"+ "\\imports-hs10\\" + "TOTAL" + "data-" + date + ".parquet"
    
    
    if os.path.exists(out_file):
        
        print("Already have downloaded file")
        
        continue
    
    print(xxx)
    
    url = surl + "&CTY_CODE=" + xxx
    
    if xxx == "":
        url = surl
    
    r = requests.get(url) 
    
    print(r)
    
    foo = pd.DataFrame(r.json()[1:]) # This then converts it to a dataframe
    # Note that the first entry is the labels

    foo.columns = r.json()[0]

    pq.write_table(pa.Table.from_pandas(foo), out_file)


<Response [200]>
5700
<Response [200]>
2010
<Response [200]>
1220
<Response [200]>
5880
<Response [200]>
4280
<Response [200]>
5800
<Response [200]>
5520
<Response [200]>
5830
<Response [200]>
4190
<Response [200]>
5330
<Response [200]>
4120
<Response [200]>
4759
<Response [200]>
4279
<Response [200]>
4419
<Response [200]>
5570
<Response [200]>
5490
<Response [200]>
3510
<Response [200]>
5590
<Response [200]>
4210
<Response [200]>
5600
<Response [200]>
5081
<Response [200]>
4231
<Response [200]>
5170
<Response [200]>
4700
<Response [200]>
4621
<Response [200]>
3010
<Response [200]>
6021
<Response [200]>
4330
<Response [200]>
4010
<Response [200]>
3370
<Response [200]>
0003
<Response [200]>
0020
<Response [200]>


In [8]:
foo.head()

,CTY_NAME,CON_VAL_MO,CAL_DUT_MO,I_COMMODITY,I_COMMODITY_SDESC,time,COMM_LVL,CTY_CODE
0,USMCA (NAFTA),364734,0,0102390061,BUFFALO FOR IMMEDIATE SLAUGHTER GT=320 KG EA N...,2013-02,HS10,0020
1,USMCA (NAFTA),588222,0,0102390061,BUFFALO FOR IMMEDIATE SLAUGHTER GT=320 KG EA N...,2013-03,HS10,0020
2,USMCA (NAFTA),584104,0,0102390061,BUFFALO FOR IMMEDIATE SLAUGHTER GT=320 KG EA N...,2013-04,HS10,0020
3,USMCA (NAFTA),93744,0,0102390061,BUFFALO FOR IMMEDIATE SLAUGHTER GT=320 KG EA N...,2013-05,HS10,0020
4,USMCA (NAFTA),740615,0,0102390061,BUFFALO FOR IMMEDIATE SLAUGHTER GT=320 KG EA N...,2013-06,HS10,0020


In [ ]:
# end_use = "hs?get=CTY_NAME,CON_VAL_MO,CAL_DUT_MO,I_COMMODITY,I_COMMODITY_SDESC,I_COMMODITY_LDESC"

# surl = "https://api.census.gov/data/timeseries/intltrade/imports/" + end_use 

# surl  = surl + my_key + "&time=" + "from+2013-01" + "&COMM_LVL=HS10" 

# url = surl
    
# r = requests.get(url) 
    
# print(r)
    
# foo = pd.DataFrame(r.json()[1:]) # This then converts it to a dataframe
#     # Note that the first entry is the labels

# foo.columns = r.json()[0]

# unique_commodities.to_csv('.\\data\\unique_commodities.csv', index=False)


ChunkedEncodingError: ('Connection broken: IncompleteRead(4089 bytes read, 6151 more expected)', IncompleteRead(4089 bytes read, 6151 more expected))